# 03 · Optimasi & Callbacks — Bab 4

*Pengantar Deep Learning untuk Meteorologi* · Kanada Kurniawan

Notebook pendamping Bab 4: mempelajari learning rate, batch size, callback, dan membaca learning curve. Prasyarat: Bab 2 (`ch-02-01_regresi_pasang_surut`).

## 1. Setup & Data Sintetik

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)
print("TensorFlow:", tf.__version__)

# data pasang surut sintetik (seperti Bab 2)
t = np.arange(0, 800)
y = np.sin(2*np.pi*t/12.42) + 0.05*np.random.randn(len(t))
X = np.column_stack([y[:-2], y[1:-1]])
target = y[2:]

n = len(X)
ntr, nva = int(n*0.7), int(n*0.15)
Xtr, ytr = X[:ntr], target[:ntr]
Xva, yva = X[ntr:ntr+nva], target[ntr:ntr+nva]
Xte, yte = X[ntr+nva:], target[ntr+nva:]
print("train", Xtr.shape, "val", Xva.shape, "test", Xte.shape)

## 2. Gradient Tape (Backpropagation Manual)

Satu langkah pelatihan dijelaskan secara manual (Kode 4.1): `GradientTape` mencatat operasi, menghitung gradien, lalu optimizer menggeser bobot.

In [ ]:
w = tf.Variable(0.5)
b = tf.Variable(0.1)
opt = tf.keras.optimizers.Adam(learning_rate=0.1)

x = tf.constant([1.0, 2.0, 3.0])
y_true = tf.constant([2.0, 4.0, 6.0])

for step in range(50):
    with tf.GradientTape() as tape:
        pred = w * x + b
        loss = tf.reduce_mean(tf.square(pred - y_true))
    grads = tape.gradient(loss, [w, b])
    opt.apply_gradients(zip(grads, [w, b]))

print("w:", w.numpy(), "| b:", b.numpy(), "| loss:", round(loss.numpy(), 5))
print("Idealnya w mendekati 2.0 dan b mendekati 0.")

## 3. Pengaruh Learning Rate

Latih model yang sama dengan learning rate berbeda dan amati loss validasi.

In [ ]:
def build_lr(eta):
    m = tf.keras.Sequential([
        tf.keras.layers.Dense(8, activation="relu", input_shape=(2,)),
        tf.keras.layers.Dense(8, activation="relu"),
        tf.keras.layers.Dense(1),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=eta), loss="mse", metrics=["mae"])
    return m

for eta in [0.01, 0.001, 0.0001]:
    m = build_lr(eta)
    h = m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=60, batch_size=32, verbose=0)
    print(f"LR={eta}: final train loss={h.history['loss'][-1]:.5f} | val loss={h.history['val_loss'][-1]:.5f}")

## 4. Callback: EarlyStopping + Checkpoint + ReduceLROnPlateau

In [ ]:
m = build_lr(0.001)
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("best_weights_ch4.keras", monitor="val_loss", save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5),
]
h = m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=400, batch_size=32,
          callbacks=callbacks, verbose=0)
print("Terlatih hingga epoch:", len(h.history["loss"]))
print("val_loss terakhir:", round(h.history["val_loss"][-1], 5))

## 5. Membaca Learning Curve

Plot train vs val loss untuk mendeteksi overfit (Gambar 4.1).

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(h.history["loss"], label="Train loss")
plt.plot(h.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("Learning curve (dengan early stopping)")
plt.legend()
plt.tight_layout()
plt.savefig("learning_curve_ch4.png", dpi=150)
plt.show()

## 6. Latihan Mini

1. Ulangi §3 dengan `tanh` sebagai aktivasi tersembunyi — bandingkan hasilnya dengan ReLU.
2. Coba `batch_size=16` vs `256`; perhatikan seberapa 'berisik' kurva loss-nya.
3. Hapus `restore_best_weights=True` dan bandingkan val_loss akhir dengan yang ada.
4. Ubah `patience` EarlyStopping menjadi 2 — apakah berhenti terlalu dini?